# Image-only static dRSA

Loads one image-session `natraster.mat` made by `first_preprocessing.m`,
selects only `img_` conditions, and compares the neural image RDM timecourse
with the final-frame model RDM for every layer.

Neural conditions remain in MATLAB's `uniqueImage` order. Model features are
explicitly reordered to that same sequence by stimulus identity.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import yaml


def find_project_root(start_path: Path) -> Path:
    '''Return the nearest parent directory containing config.yaml.'''
    for path in (start_path, *start_path.parents):
        if (path / "config.yaml").exists():
            return path
        # end if config.yaml exists
    # end for path
    raise FileNotFoundError("Could not find config.yaml from the notebook directory.")
# EOF


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
ENV = os.getenv("MY_ENV", "tiziano_mac_mini")

with open(PROJECT_ROOT / "config.yaml", "r") as file:
    config = yaml.safe_load(file)
# end with open

paths = config[ENV]["paths"]
sys.path.extend([paths["src_path"], paths["useful_stuff_path"]])

from image_processing.video_feature_extraction import (
    list_video_feature_files,
    load_aligned_video_features,
    match_feature_stimulus_names,
)
from project_specific_utils import (
    load_natraster,
    min_max_normalization,
    select_stimulus_rasters,
)
from useful_stuff.general_utils import TimeSeries, create_RDM
from useful_stuff.general_utils.RSA import dRSA


In [ ]:
@dataclass
class Cfg:
    # Replace this placeholder with the image-only recording session name.
    exp_name: str = "UPDATE_WITH_IMAGE_SESSION"

    # MATLAB channels 84--186 inclusive; Python stop indices are exclusive.
    good_channels: tuple[int, int] = (84, 186)
    # Optional zero-based slice relative to good_channels; None keeps all of them.
    analysis_channels: tuple[int, int] | None = (17, 25)
    image_crop_ms: int = 1000
    new_fs: float | None = 100
    normalization: str | None = None  # Supported: None or "min_max".

    signal_RDM_metric: str = "cosine_cnt"
    model_RDM_metric: str = "cosine_cnt"
    RSA_metric: str = "spearman"

    model_name: str = "ijepa_vith14_1k"
    model_dataset_name: str = "static_dynamic"
    model_pooling: str | None = "mean"
    model_features_dir: Path | None = None
    model_frame_index: int = -1


cfg = Cfg()
natraster_mat_path = (
    Path(paths["data_path"]) / "data" / f"{cfg.exp_name}_natraster.mat"
)
cfg


In [ ]:
# Select image conditions without requiring corresponding video trials.
rasters, all_stimulus_names = load_natraster(natraster_mat_path)
image_rasters, image_names = select_stimulus_rasters(
    rasters, all_stimulus_names, stimulus_prefix="img_",
)

# Convert MATLAB's one-based inclusive channel range to a Python slice.
first_channel, last_channel = cfg.good_channels
channel_slice = slice(first_channel - 1, last_channel)
image_rasters = image_rasters[channel_slice, :cfg.image_crop_ms, :]

if cfg.analysis_channels is not None:
    first_analysis_channel, last_analysis_channel = cfg.analysis_channels
    image_rasters = image_rasters[
        first_analysis_channel:last_analysis_channel, :, :
    ]
# end if cfg.analysis_channels

image_neural_ts = TimeSeries(image_rasters, fs=1000)
if cfg.new_fs is not None:
    image_neural_ts.resample(cfg.new_fs)
# end if cfg.new_fs

if cfg.normalization == "min_max":
    image_neural_ts.set_array(
        min_max_normalization(image_neural_ts.get_array())
    )
elif cfg.normalization is not None:
    raise ValueError("cfg.normalization must be None or 'min_max'")
# end if cfg.normalization

print(f"Image conditions: {len(image_names)}")
print(f"Neural raster: {image_neural_ts.shape()} (channels, time, stimuli)")
print(f"Sampling frequency: {image_neural_ts.get_fs()} Hz")
print("MATLAB order retained:", image_names[:3], "...", image_names[-3:])


In [ ]:
# Show both the population timecourse and the channel-by-time response map.
image_time_ms = (
    np.arange(image_neural_ts.shape()[1]) * 1000 / image_neural_ts.get_fs()
)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
axes[0].plot(
    image_time_ms,
    image_neural_ts.get_array().mean(axis=(0, 2)),
    color="tab:blue",
)
axes[0].set(
    xlabel="Time from image onset (ms)",
    ylabel="Mean response",
    title="Image population response",
)
response_image = axes[1].imshow(
    image_neural_ts.get_array().mean(axis=2),
    aspect="auto",
    origin="lower",
    extent=[image_time_ms[0], image_time_ms[-1], 0, image_neural_ts.shape()[0]],
)
axes[1].set(
    xlabel="Time from image onset (ms)",
    ylabel="Selected channel",
    title="Mean response across image conditions",
)
fig.colorbar(response_image, ax=axes[1], label="Response")


## Align model features

The matcher ignores `img_`/`vid_` and the filename extension, then returns
exact HDF5 dataset names in the unchanged neural/MATLAB order. This permits
image trials named `img_example.jpg` to use the final frame of
`vid_example.mp4` features.


In [ ]:
model_features_dir = (
    cfg.model_features_dir or Path(paths["data_path"]) / "models"
)
feature_paths = list_video_feature_files(
    model_features_dir,
    cfg.model_name,
    cfg.model_dataset_name,
    cfg.model_pooling,
)
image_feature_names = match_feature_stimulus_names(
    feature_paths[0], image_names,
)

print(f"Model layers: {len(feature_paths)}")
print("First neural-to-model mappings:")
for neural_name, feature_name in zip(image_names[:5], image_feature_names[:5]):
    print(f"  {neural_name} -> {feature_name}")
# end for neural_name, feature_name


In [ ]:
# Compute the neural image RDM once because it is shared by every model layer.
image_drsa = dRSA(
    signal_RDM_metric=cfg.signal_RDM_metric,
    model_RDM_metric=cfg.model_RDM_metric,
    RSA_metric=cfg.RSA_metric,
)
image_drsa.compute_RDM_timeseries(image_neural_ts, "signal")

image_layer_drsa = []
layer_names = []
for feature_path in feature_paths:
    final_frame_features, layer_name, _ = load_aligned_video_features(
        feature_path,
        image_feature_names,
        frame_index=cfg.model_frame_index,
    )
    model_RDM = create_RDM(
        final_frame_features, metric=cfg.model_RDM_metric
    )
    image_drsa.set_RDM(model_RDM, "model")
    image_layer_drsa.append(
        image_drsa.compute_static_dRSA().get_array().copy()
    )
    layer_names.append(layer_name)
# end for feature_path

image_layer_drsa = np.stack(image_layer_drsa, axis=0)
print(f"Static dRSA: {image_layer_drsa.shape} (layers, neural time)")


In [ ]:
image_time_ms = (
    np.arange(image_layer_drsa.shape[1]) * 1000 / image_neural_ts.get_fs()
)
layer_colors = plt.cm.plasma(np.linspace(0, 1, len(layer_names)))

fig, ax = plt.subplots(figsize=(13, 6))
for layer_index, (layer_name, layer_drsa) in enumerate(
    zip(layer_names, image_layer_drsa)
):
    ax.plot(
        image_time_ms,
        layer_drsa,
        color=layer_colors[layer_index],
        linewidth=1.4,
        label=layer_name,
    )
# end for layer_index
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set(
    xlabel="Time from image onset (ms)",
    ylabel=f"Static dRSA ({cfg.RSA_metric})",
    title=f"{cfg.model_name}: final-frame model RDM vs neural image RDM",
)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, ncol=2)
fig.tight_layout()
